<a href="https://colab.research.google.com/github/UmuhireJessie/diabetes-project/blob/main/tabnet_vs_ei_comparison_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# pytorch-tabnet is the key addition — all other packages match notebook 1
!pip install scikit-learn imblearn xgboost pandas pytorch-tabnet matplotlib seaborn statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.3 MB/s eta 0:00:00


In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/eumubyey/pond-dataset/')
data_dir       = '/kaggle/input/datasets/eumubyey/pond-dataset/diabetes/diabetes/POND_data/'
ei_results_dir = '/kaggle/input/datasets/eumubyey/pond-dataset/diabetes/diabetes/results/'
ei_models_dir  = '/kaggle/input/datasets/eumubyey/pond-dataset/diabetes/diabetes/'

In [ ]:
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sklearn utilities
from sklearn import metrics
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

# Class imbalance
from imblearn.under_sampling import RandomUnderSampler

# Statistical testing
from scipy.stats import ranksums
from statsmodels.stats.multitest import fdrcorrection

# TabNet
from pytorch_tabnet.tab_model import TabNetClassifier

from utils import fmeasure_score
import torch


print('All imports successful.')

All imports successful.


In [ ]:
import huggingface_hub
huggingface_hub.login()

In [ ]:
data_dir = '/kaggle/input/datasets/eumubyey/pond-dataset/diabetes/diabetes/POND_data/'

# Data dictionary drives imputation type and modality assignment downstream
data_dict = pd.read_csv(data_dir + 'data_dictionary.csv')

# Main feature table
nhanes_data = pd.read_csv(data_dir + 'Pond_bivariate.csv')

# These three variables are excluded from modelling.
# bmipct is retained separately for the ADA/AAP rule evaluation below.
exclude_vars = ['bmipct', 'INDFMPIR', 'SDDSRVYR']

nhanes_index = nhanes_data['SEQN'].values

# Merge preDM2 labels from the cohort filev
preDM2_status = pd.read_csv(data_dir + 'Pond_9918_cohortAll.csv')
preDM2_status = preDM2_status[['SEQN', 'preDM2']]
preDM2_status = preDM2_status[preDM2_status['SEQN'].isin(nhanes_index)]

preDM2_y = nhanes_data.merge(
    preDM2_status, on='SEQN', how='left'
)['preDM2'].values.astype(int)

# Drop label and ID columns — features only from here on
nhanes_data.drop(columns=['preDM2', 'SEQN'] + exclude_vars, inplace=True, errors='ignore')

# ---- ADA/AAP clinical guideline (reference baseline) ----
# A child is flagged if BMI ≥ 85th percentile AND at least one additional risk factor.
# This is a hard binary rule — it does not produce probabilities.
predm_ada = pd.read_csv(os.path.join(data_dir, 'Pond_aap.csv'))
predm_ada['ada'] = (
    (predm_ada['bmipct'] >= 85) &
    (predm_ada['HbpAAP'] | predm_ada['ChlAAP'] | predm_ada['raceAAP'])
).astype(float)

intersect_set = set(predm_ada['SEQN']).intersection(set(nhanes_index))
predm_ada_intersects = predm_ada[predm_ada['SEQN'].isin(intersect_set)]
predm_ada_intersects = predm_ada_intersects.merge(preDM2_status, on='SEQN', how='left')

print(f'Feature matrix : {nhanes_data.shape}')
print(f'Label counts   : {pd.Series(preDM2_y).value_counts().to_dict()}  (0=no preDM2, 1=preDM2)')
print(f'Class imbalance ratio: {preDM2_y.sum() / len(preDM2_y):.2%} positive')

Feature matrix : (15149, 95)
Label counts   : {0: 13139, 1: 2010}  (0=no preDM2, 1=preDM2)
Class imbalance ratio: 13.27% positive


In [ ]:
modalities = data_dict['Domain'].dropna().unique().tolist()
modalities_mapping = {}   # will hold one DataFrame per domain
columns_na_mapping = {}   # stores fill values (useful for inference later)

# ---- Step 1: Imputation ----
# Fill missing values based on the variable type recorded in the data dictionary.
# This is done BEFORE encoding so mode/median is computed on the raw values.
for c in nhanes_data.columns:
    col_type = data_dict.loc[data_dict['VariableNameNHANES'] == c, 'Type'].values
    if len(col_type) == 0:
        val = nhanes_data[c].median()
    elif col_type[0] in ('Categorical', 'Binary'):
        val = nhanes_data[c].mode().values[0]
    else:  # Continuous or Ordinal
        val = nhanes_data[c].median()
    nhanes_data[c] = nhanes_data[c].fillna(val)
    columns_na_mapping[c] = val

# ---- Step 2: One-hot encoding, per modality ----
# Only categorical columns with >2 unique values are expanded.
# Binary columns are left as 0/1. Continuous/ordinal columns are left as-is.
data_dict_filtered = data_dict[~data_dict['VariableNameNHANES'].isin(exclude_vars)]

for m in modalities:
    col_names = data_dict_filtered.loc[
        data_dict_filtered['Domain'] == m, 'VariableNameNHANES'
    ].unique().tolist()
    dfs_ = []
    for c in col_names:
        col_type = data_dict_filtered.loc[
            data_dict_filtered['VariableNameNHANES'] == c, 'Type'
        ].values
        if len(col_type) > 0 and col_type[0] == 'Categorical' and nhanes_data[c].nunique() > 2:
            dfs_.append(pd.get_dummies(nhanes_data[c].astype(int), prefix=c))
        else:
            dfs_.append(nhanes_data[[c]])
    modalities_mapping[m] = pd.concat(dfs_, axis=1)

# Concatenate all modalities into one flat feature matrix for TabNet.
X_all = pd.concat(list(modalities_mapping.values()), axis=1)
feature_names = X_all.columns.tolist()

print(f'Full feature matrix (all modalities): {X_all.shape}')
for m, df in modalities_mapping.items():
    print(f'  {m}: {df.shape[1]} features')

Full feature matrix (all modalities): (15149, 108)
  Sociodemographic: 37 features
  Health status: 17 features
  Other lifestyle behaviors: 6 features
  Diet: 48 features


In [ ]:
def metrics_ADA(labels, predictions):
    """Evaluate a hard binary rule (e.g. ADA/AAP guideline).
    The threshold is implicit in the 0/1 predictions — no optimisation is applied.
    Used only for the clinical baseline.
    """
    perf_dict = {'metrics': [], 'scores': [], 'class': []}
    frp_pos = fmeasure_score(labels, predictions, pos_label=1)
    frp_neg = fmeasure_score(labels, predictions, thres=1 - frp_pos['thres'], pos_label=0)
    entries = [
        ('F-measure (pos)', frp_pos['F'], 'pos'),
        ('PPV',             frp_pos['P'], 'pos'),
        ('Sensitivity',     frp_pos['R'], 'pos'),
        ('F-measure (neg)', frp_neg['F'], 'neg'),
        ('NPV',             frp_neg['P'], 'neg'),
        ('Specificity',     frp_neg['R'], 'neg'),
        ('BalancedACC',     (frp_neg['R'] + frp_pos['R']) / 2, 'NA'),
        ('AUC',             metrics.roc_auc_score(labels, predictions), 'NA'),
    ]
    for name, score, cls in entries:
        perf_dict['metrics'].append(name)
        perf_dict['scores'].append(score)
        perf_dict['class'].append(cls)
    return pd.DataFrame(perf_dict)


def metrics_BA(labels, predictions):
    """Evaluate a probabilistic classifier.
    Sweeps the ROC curve to find the threshold maximising (sensitivity + specificity) / 2,
    then reports all metrics at that threshold.
    This is identical to notebook 1 — do not modify.
    """
    perf_dict = {'metrics': [], 'scores': [], 'class': []}
    fpr, tpr, thresholds = metrics.roc_curve(labels, predictions)
    specificity = 1 - fpr
    balance_accs = (specificity + tpr) / 2
    max_thres = thresholds[np.argmax(balance_accs)]

    frp_pos = fmeasure_score(labels, predictions, pos_label=1, thres=max_thres)
    frp_neg = fmeasure_score(labels, predictions, thres=1 - max_thres, pos_label=0)

    entries = [
        ('F-measure (pos)', frp_pos['F'], 'pos'),
        ('PPV',             frp_pos['P'], 'pos'),
        ('Sensitivity',     frp_pos['R'], 'pos'),
        ('F-measure (neg)', frp_neg['F'], 'neg'),
        ('NPV',             frp_neg['P'], 'neg'),
        ('Specificity',     frp_neg['R'], 'neg'),
        ('BalancedACC',     max(balance_accs), 'NA'),
        ('AUC',             metrics.roc_auc_score(labels, predictions), 'NA'),
    ]
    for name, score, cls in entries:
        perf_dict['metrics'].append(name)
        perf_dict['scores'].append(score)
        perf_dict['class'].append(cls)
    df = pd.DataFrame(perf_dict)
    df['optimize'] = 'BalanceAcc'
    return df


# Compute the ADA/AAP baseline scores now — used as a reference line in plots
ada_scores = metrics_ADA(
    labels=predm_ada_intersects['preDM2'],
    predictions=predm_ada_intersects['ada']
)
ada_scores

,metrics,scores,class
0,F-measure (pos),0.255451,pos
1,PPV,0.177011,pos
2,Sensitivity,0.458730,pos
3,F-measure (neg),0.768123,neg
4,NPV,0.891101,neg
5,Specificity,0.674972,neg
6,BalancedACC,0.566851,NA
7,AUC,0.566851,NA


In [ ]:
# TabNet constructor arguments — passed at model creation
TABNET_PARAMS = dict(
    n_d=16,           # width of the prediction layer embedding
    n_a=16,           # width of the attention embedding (usually equal to n_d)
    n_steps=3,        # number of sequential attention steps
    gamma=1.5,        # feature re-usage coefficient across steps
    n_independent=1,  # step-specific GLU layers
    n_shared=2,       # shared GLU layers
    momentum=0.02,    # BN momentum
    optimizer_params=dict(lr=1e-4, weight_decay=1e-3),
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    scheduler_params=dict(step_size=50, gamma=0.9),
    verbose=0,
)

# TabNet fit arguments — passed at training time
FIT_PARAMS = dict(
    max_epochs=500,
    patience=50,           # early stopping patience — stops if val AUC doesn't improve
    batch_size=128,
    virtual_batch_size=64,  # ghost batch normalisation size
    eval_metric=['auc'],     # monitor AUC on the validation fold during training
)

# Reproduce the exact same 10 seeds used in notebook 1
random.seed(41)
random_seeds = random.sample(range(100), 10)
print(f'Random seeds (same as notebook 1): {random_seeds}')

Random seeds (same as notebook 1): [48, 42, 29, 21, 49, 73, 88, 36, 70, 35]


In [ ]:
X_np = X_all.values.astype(np.float32)
y_np = preDM2_y.astype(int)

# Feature selection
ei_imp = pd.read_csv(ei_results_dir + 'int.csv')
top_features = ei_imp['feature'].head(40).tolist()
X_filtered = X_all[[f for f in top_features if f in X_all.columns]]

# Override X_np with the filtered version
X_np = X_filtered.values.astype(np.float32)
feature_names = X_filtered.columns.tolist()

print(f'Original feature matrix: {X_all.shape}')
print(f'Filtered feature matrix: {X_np.shape}')

Seed 1/10 (seed=48)

Early stopping occurred at epoch 54 with best_epoch = 34 and best_val_0_auc = 0.60898

Early stopping occurred at epoch 39 with best_epoch = 19 and best_val_0_auc = 0.59784

Early stopping occurred at epoch 37 with best_epoch = 17 and best_val_0_auc = 0.62702

Early stopping occurred at epoch 85 with best_epoch = 65 and best_val_0_auc = 0.62186

Early stopping occurred at epoch 66 with best_epoch = 46 and best_val_0_auc = 0.58771
  → AUC=0.6074  BalancedACC=0.5860
Seed 2/10 (seed=42)

Early stopping occurred at epoch 70 with best_epoch = 50 and best_val_0_auc = 0.62125

Early stopping occurred at epoch 59 with best_epoch = 39 and best_val_0_auc = 0.63717

Early stopping occurred at epoch 79 with best_epoch = 59 and best_val_0_auc = 0.61644

Early stopping occurred at epoch 73 with best_epoch = 53 and best_val_0_auc = 0.63027

Early stopping occurred at epoch 52 with best_epoch = 32 and best_val_0_auc = 0.61021
  → AUC=0.6173  BalancedACC=0.5836
Seed 3/10 (seed=29)


In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Retrieve the secret you set
secret_client = UserSecretsClient()
hf_token = secret_client.get_secret("HF_TOKEN")

# Log in to Hugging Face
login(token=hf_token)
print("Successfully logged in with HF_TOKEN.")

TabPFN loaded successfully.


In [ ]:
!pip install tabpfn -q

from tabpfn import TabPFNClassifier

tabpfn_results = []

for idx, seed in enumerate(random_seeds):
    print(f'Seed {idx+1}/10 (seed={seed})')
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    oof_probs  = np.zeros(len(y_np))
    oof_labels = np.zeros(len(y_np), dtype=int)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_np, y_np)):
        X_tr, y_tr   = X_np[train_idx], y_np[train_idx]
        X_val, y_val = X_np[val_idx],   y_np[val_idx]

        # TabPFN has a 10,000 sample training limit
        # If training fold exceeds this, undersample to fit within the limit
        if len(X_tr) > 10000:
            rus = RandomUnderSampler(random_state=seed)
            X_tr, y_tr = rus.fit_resample(X_tr, y_tr)

        # Scale within fold — same as TabNet
        scaler = StandardScaler()
        X_tr_sc  = scaler.fit_transform(X_tr).astype(np.float32)
        X_val_sc = scaler.transform(X_val).astype(np.float32)

        # No training loop, no hyperparameters — just fit and predict
        clf = TabPFNClassifier(device='cuda', n_estimators=32)
        clf.fit(X_tr_sc, y_tr)

        oof_probs[val_idx]  = clf.predict_proba(X_val_sc)[:, 1]
        oof_labels[val_idx] = y_val
        print(f'  fold {fold+1} done')

    df_seed = metrics_BA(oof_labels, oof_probs)
    df_seed['dummy']   = idx
    df_seed['dataset'] = 'TabPFN'
    tabpfn_results.append(df_seed)

    auc = df_seed.loc[df_seed['metrics'] == 'AUC', 'scores'].values[0]
    ba  = df_seed.loc[df_seed['metrics'] == 'BalancedACC', 'scores'].values[0]
    print(f'  → AUC={auc:.4f}  BalancedACC={ba:.4f}')

tabpfn_df = pd.concat(tabpfn_results, ignore_index=True)
print('\nTabPFN complete.')

---
## Section 8: Load EI Results from Notebook 1

Load the pre-computed EI performance CSVs saved by notebook 1.
These are the score distributions we compare TabNet against.
Loading from CSV rather than re-running EI ensures exact reproducibility.

In [ ]:
# These CSVs were saved at the end of notebook 1.
# Each file has 10 rows (one per seed) and columns for each model/modality.
ei_auc = pd.read_csv(ei_results_dir + 'nhanes_all_AUC.csv')
ei_ba  = pd.read_csv(ei_results_dir + 'nhanes_all_BA.csv')

# The best-performing EI ensemble method from notebook 1 was S.LR (stacked logistic regression).
# We use that as the primary EI comparator.
EI_BEST_METHOD = 'S.LR'

# Reshape EI scores into long format to match tabnet_all_df structure
def ei_to_long(ei_wide, metric_name):
    """Convert the wide-format EI CSV (rows=seeds, cols=datasets) to long format."""
    rows = []
    for col in ei_wide.columns:
        if col == 'dummy':
            continue
        for i, val in enumerate(ei_wide[col].values):
            rows.append({'metrics': metric_name, 'scores': val,
                         'dataset': col, 'dummy': i, 'optimize': 'BalanceAcc'})
    return pd.DataFrame(rows)

ei_auc_long = ei_to_long(ei_auc, 'AUC')
ei_ba_long  = ei_to_long(ei_ba,  'BalancedACC')
ei_long = pd.concat([ei_auc_long, ei_ba_long], ignore_index=True)

print('EI results loaded.')
print(ei_auc.head())

In [ ]:
# Combine TabNet (all modalities) + TabNet (per modality) + EI
all_results = pd.concat([
    # tabnet_all_df,
    tabpfn_df,
    # tabnet_mod_df,
    ei_long
], ignore_index=True)

# Summary table: mean ± std per (dataset, metric)
summary = (
    all_results[all_results['metrics'].isin(['AUC', 'BalancedACC'])]
    .groupby(['dataset', 'metrics'])['scores']
    .agg(['mean', 'std'])
    .round(4)
    .rename(columns={'mean': 'Mean', 'std': 'Std'})
)

print('Performance summary (mean ± std across 10 seeds, 5-fold CV):')
summary

In [ ]:
# ---- Select which models/modalities to show ----
# For EI, use the best-performing ensemble method (S.LR) to keep the plot clean.
# For TabNet, include the all-modalities run and each individual modality.

plot_datasets = [
    'TabPFN',
    'EI\\n(multi-domain)'
]

# Rename for readability in the plot
rename_map = {
    'BalancedACC': 'Balanced Accuracy',
}
plot_df = all_results.replace(rename_map)
ada_scores_plot = ada_scores.replace(rename_map)

color_map = ['#1AA2E7', '#1D1F68']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, metric in zip(axes, ['AUC', 'Balanced Accuracy']):
    raw_metric = 'BalancedACC' if metric == 'Balanced Accuracy' else metric
    metric_df  = plot_df[plot_df['metrics'].isin([metric, raw_metric])]
    ada_val    = ada_scores_plot.loc[
        ada_scores_plot['metrics'].isin([metric, raw_metric]), 'scores'
    ].values

    order = [
        'TabPFN',
        'EI\n(multi-domain)',
    ]

    sns.barplot(
        data=metric_df, x='scores', y='dataset',
        order=[o for o in order if o in metric_df['dataset'].values],
        palette=color_map, ax=ax, errorbar='sd'
    )

    # ADA/AAP reference line — the clinical threshold both models aim to beat
    if len(ada_val):
        ax.axvline(ada_val[0], color='red', linestyle='--', linewidth=1.5, label='ADA/AAP')
        ax.legend(fontsize=11)

    ax.set_xlabel(metric, fontsize=13, fontweight='bold')
    ax.set_ylabel('', fontsize=12)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    for tick in ax.yaxis.get_major_ticks():
        tick.label1.set_fontsize(12)
        tick.label1.set_fontweight('bold')

plt.suptitle('TabPFN vs Ensemble Integration — Prediabetes Prediction\n(error bars = ±1 SD across 10 seeds)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('tabpfn_vs_ei_performance.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
def pairwise_rank_sum(df):
    """Pairwise Wilcoxon rank-sum tests with FDR correction.
    Tests the one-sided hypothesis: row model > column model.
    Identical to the function in notebook 1.
    """
    groups = df.columns.tolist()
    n = len(groups)
    pairwise_mat = -1 * np.ones((n, n))
    idx_list, pval_list = [], []

    for i, a in enumerate(groups):
        for j, b in enumerate(groups):
            if i != j:
                idx_list.append([i, j])
                pval_list.append(ranksums(df[a].values, df[b].values, alternative='greater')[1])

    # FDR correction to account for multiple comparisons
    corrected = fdrcorrection(pval_list)[1]
    for (i, j), p in zip(idx_list, corrected):
        pairwise_mat[i, j] = p

    return pd.DataFrame(pairwise_mat, index=groups, columns=groups)


def build_score_matrix(results_df, metric):
    """Pivot results into wide format (rows=seeds, cols=models) for statistical testing."""
    return (
        results_df[results_df['metrics'] == metric]
        .pivot_table(index='dummy', columns='dataset', values='scores', aggfunc='mean')
    )


# Build score matrices for AUC and Balanced Accuracy
# Include TabNet (all modalities) and the best EI configuration
comparison_df = pd.concat([
    tabpfn_df,
    ei_long[ei_long['dataset'] == EI_BEST_METHOD].assign(dataset='EI (S.LR)')
], ignore_index=True)

auc_matrix = build_score_matrix(comparison_df, 'AUC')
ba_matrix  = build_score_matrix(comparison_df, 'BalancedACC')

print('=== Pairwise Wilcoxon rank-sum p-values (FDR-corrected) — AUC ===')
print('Values < 0.05 indicate row model significantly outperforms column model.')
pairwise_rank_sum(auc_matrix)

In [ ]:
print('=== Pairwise Wilcoxon rank-sum p-values (FDR-corrected) — Balanced Accuracy ===')
pairwise_rank_sum(ba_matrix)

In [ ]:
# ---- Retrain on full data for feature importance ----
# We use the full dataset here (not CV) solely to get stable attention weights.
# Performance metrics above should not be drawn from this model.

scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X_np).astype(np.float32)

# Undersample for the full-data fit
rus_full = RandomUnderSampler(random_state=42)
X_full_res, y_full_res = rus_full.fit_resample(X_full_scaled, y_np)

clf_full = TabNetClassifier(seed=42, **TABNET_PARAMS)
clf_full.fit(
    X_full_res, y_full_res,
    max_epochs=200,
    patience=20,
    batch_size=256,
    virtual_batch_size=128,
)

# feature_importances_ aggregates attention weights across all steps and all training samples
importances = clf_full.feature_importances_
imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).reset_index(drop=True)

print(f'Top 10 features by TabNet attention importance:')
imp_df.head(10)

In [ ]:
# ---- Global feature importance plot (top 30) ----
# Compare this ranking against the EI rank-product scores from notebook 1
# to assess whether both methods agree on the most predictive features.

fig, ax = plt.subplots(figsize=(9, 9))
top30 = imp_df.head(30)

sns.barplot(
    data=top30, x='importance', y='feature',
    color='#1AA2E7', ax=ax
)
ax.set_title('Top-30 Features — TabNet Attention Importance\n(full-data model, all modalities)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Aggregated Attention Weight', fontsize=12)
ax.set_ylabel('')
for tick in ax.yaxis.get_major_ticks():
    tick.label1.set_fontsize(11)

plt.tight_layout()
plt.savefig('tabnet_feature_importance_global.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Get per-instance attention masks from the full-data model.
# explain() returns masks of shape (n_samples, n_features) — one weight per feature per sample.
explain_matrix, _ = clf_full.explain(X_full_scaled)

# Identify one high-risk and one low-risk individual from the original (unsampled) data
# for illustrative purposes.
proba_full = clf_full.predict_proba(X_full_scaled)[:, 1]
high_risk_idx = np.argmax(proba_full)    # individual with highest predicted preDM2 probability
low_risk_idx  = np.argmin(proba_full)    # individual with lowest predicted preDM2 probability

def plot_instance_importance(idx, label, ax, top_n=15):
    """Plot attention weights for a single individual."""
    weights = explain_matrix[idx]
    instance_imp = pd.DataFrame({'feature': feature_names, 'weight': weights})
    instance_imp = instance_imp.sort_values('weight', ascending=False).head(top_n)
    sns.barplot(data=instance_imp, x='weight', y='feature', color='#1AA2E7', ax=ax)
    ax.set_title(f'{label}\n(predicted prob={proba_full[idx]:.3f})',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Attention weight', fontsize=10)
    ax.set_ylabel('')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_instance_importance(high_risk_idx, 'High-risk individual', axes[0])
plot_instance_importance(low_risk_idx,  'Low-risk individual',  axes[1])

plt.suptitle('Instance-level TabNet attention — which features drove each prediction?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('tabnet_instance_importance.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Load EI feature importance from notebook 1 (saved as int.csv)
ei_imp = pd.read_csv(ei_results_dir + 'int.csv')

# Standardise column names for comparison
# Notebook 1 saves rank-product scores — lower rank = more important
ei_top30  = set(ei_imp['feature'].head(30).tolist())
tab_top30 = set(imp_df['feature'].head(30).tolist())

overlap = ei_top30.intersection(tab_top30)
print(f'Features in top-30 of BOTH TabNet and EI: {len(overlap)}/30')
print('\nOverlapping features:')
for f in sorted(overlap):
    tab_rank = imp_df[imp_df['feature']==f].index[0] + 1
    ei_rank  = ei_imp[ei_imp['feature']==f].index[0] + 1 if 'feature' in ei_imp.columns else 'N/A'
    print(f'  {f:<40} TabNet rank: {tab_rank:>3}   EI rank: {ei_rank:>3}')

In [ ]:
# Save TabPFN results
tabpfn_auc_wide = (
    tabpfn_df[tabpfn_df['metrics'] == 'AUC']
    .pivot(index='dummy', columns='dataset', values='scores')
)
tabpfn_auc_wide.to_csv('tabpfn_AUC.csv')

tabpfn_ba_wide = (
    tabpfn_df[tabpfn_df['metrics'] == 'BalancedACC']
    .pivot(index='dummy', columns='dataset', values='scores')
)
tabpfn_ba_wide.to_csv('tabpfn_BA.csv')

# TabNet feature importance (from full-data model in Section 12)
imp_df.to_csv('tabnet_feature_importance.csv', index=False)

print('Saved: tabpfn_AUC.csv, tabpfn_BA.csv, tabnet_feature_importance.csv')
print('\nFinal summary:')
print(summary)